# Version 6 recovery: validation → public-test inference

Select GPU T4 (T4 ×2 also works; inference uses GPU 0), enable Internet and Run All.
Input `cymeu4l0t/lifemate-v6-completed-adapters` must use dataset version 2 or later; version 1 omitted the adapter directories. Refresh/update this input in the Kaggle editor before starting an existing session.
No training. Exact old Vietnamese prompt policy and old JSON tool presentation from the completed version-6 training experiment. No English/TypeScript or anti-repeat policy changes.

**Default mode: trained Planner; Caller and Summarizer use the completed Global adapter.** This is an explicit fallback configuration: those two local adapters were never trained. Optional MODE='single' uses Global for the single-agent workflow.

Use supplied Zipformer2 ONNX ASR for current and history audio, no oracle transcript fallback. Evaluate all 100 split.txt IDs first, then predict every public_test ID with no gold scoring, and write public_test/submission.jsonl. Every executed call (including rejected calls) is retained through ToolEnv. Models are loaded in a fresh subprocess per split to release CUDA memory and import the correct split runtime.

Completed four-sample batches are saved atomically and can be reused if the same output directory is preserved. If timeout interrupts a batch before its calls are recorded, an active-batch marker stops automatic retries; it cannot silently restore lost in-memory state or resume an arbitrary killed process. Preserve outputs when moving between Kaggle sessions. No private-test labels or automatic leaderboard submission.


In [ ]:
from pathlib import Path
import os,sys,json,hashlib,subprocess,zipfile,time,shutil
MODEL='Qwen/Qwen3-4B-Instruct-2507'
REVISION='cdbee75f17c01a7cc42f958dc650907174af0554'
MODE='multi_fallback'  # trained Planner, Global adapter for Caller/Summarizer
assert MODE in ('multi_fallback','single')
RUN_NAME='v6_planner_global_fallback_v1'
SESSION_STARTED=time.monotonic()
SESSION_HOURS=10  # Stop before new batches after this budget; no per-sample timeout.
RESUME_DIR=None  # Optional mounted prior run directory containing config.json and split subfolders.
WORK=Path('/kaggle/working/v6_validation_test')/RUN_NAME;WORK.mkdir(parents=True,exist_ok=True)
if RESUME_DIR is not None:shutil.copytree(Path(RESUME_DIR),WORK,dirs_exist_ok=True)
SOURCE=Path('/kaggle/working/v6_validation_test_source')
CORE_REV='4894d331692293565635454a8c686b7c18314a04'
if not SOURCE.exists():subprocess.run(['git','clone','--filter=blob:none','--no-checkout','https://github.com/tnisl/baseline.git',str(SOURCE)],check=True)
subprocess.run(['git','-C',str(SOURCE),'fetch','origin',CORE_REV],check=True)
subprocess.run(['git','-C',str(SOURCE),'checkout','--detach','FETCH_HEAD'],check=True)


In [ ]:
PAYLOAD={'requirements_asr_2gpu.lock': 'cffi==2.1.1\nnumpy==2.2.6\npycparser==3.12\nscipy==1.16.2\nsherpa-onnx==2.0.1\nsherpa-onnx-core==2.0.1\nsoundfile==0.13.1\n', 'requirements_infer_2gpu.lock': '--extra-index-url https://download.pytorch.org/whl/cu128\naccelerate==1.15.0\nagent-detector==2.0.0\naiohappyeyeballs==2.7.1\naiohttp==3.14.5\naiosignal==1.4.0\nairportsdata==20261030\nannotated-doc==0.0.5\nannotated-types==0.8.0\nanyio==4.16.0\nastor==0.8.1\nattrs==26.1.0\nblake3==1.0.11\ncachetools==7.2.8\ncertifi==2026.10.23\ncharset-normalizer==3.5.2\nclick==8.5.0\ncloudpickle==3.1.2\ncompressed-tensors==0.10.2\ncuda-pathfinder==1.8.5\ncupy-cuda12x==14.2.0\ndepyf==0.18.0\ndetect-installer==0.2.1\ndill==0.4.1\ndiskcache==5.6.3\ndistro==1.9.0\ndnspython==2.9.1\neinops==0.8.2\nemail-validator==2.3.0\nfastapi==0.162.0\nfastapi-cli==0.0.32\nfastapi-cloud-cli==0.26.0\nfastar==0.12.0\nfilelock==4.2.7\nfrozenlist==1.8.0\nfsspec==2026.9.0\ngguf==0.19.0\ngoogleapis-common-protos==1.76.0\nh11==0.16.0\nhf-xet==1.8.0\nhttpcore==1.0.9\nhttpcore2==2.16.0\nhttptools==0.9.0\nhttpx==0.28.1\nhttpx2==2.16.0\nhuggingface-hub==0.35.3\nidna==3.20\ninteregular==0.3.3\njinja2==3.1.6\njiter==0.18.0\njsonschema==4.25.1\njsonschema-specifications==2025.9.1\nlark==1.2.2\nllguidance==0.7.30\nllvmlite==0.44.0\nlm-format-enforcer==0.10.12\nmarkdown-it-py==4.2.0\nmarkupsafe==3.0.4\nmdurl==0.1.2\nmistral-common==1.13.1\nmpmath==1.3.0\nmsgpack==1.2.4\nmsgspec==0.22.0\nmultidict==7.2.1\nnest-asyncio==1.6.0\nnetworkx==3.7\nninja==1.13.2\nnumba==0.61.2\nnumpy==2.2.6\nnvidia-cublas-cu12==12.8.3.14\nnvidia-cuda-cupti-cu12==12.8.57\nnvidia-cuda-nvrtc-cu12==12.8.61\nnvidia-cuda-runtime-cu12==12.8.57\nnvidia-cudnn-cu12==9.7.1.26\nnvidia-cufft-cu12==11.3.3.41\nnvidia-cufile-cu12==1.13.0.11\nnvidia-curand-cu12==10.3.9.55\nnvidia-cusolver-cu12==11.7.2.55\nnvidia-cusparse-cu12==12.5.7.53\nnvidia-cusparselt-cu12==0.6.3\nnvidia-nccl-cu12==2.26.2\nnvidia-nvjitlink-cu12==12.8.61\nnvidia-nvtx-cu12==12.8.55\nopenai==1.90.0\nopencv-python-headless==5.0.0.93\nopentelemetry-api==1.45.2\nopentelemetry-exporter-http-transport==0.66b2\nopentelemetry-exporter-otlp-common==0.66b2\nopentelemetry-exporter-otlp-proto-common==1.45.2\nopentelemetry-exporter-otlp-proto-http==1.45.2\nopentelemetry-proto==1.45.2\nopentelemetry-sdk==1.45.2\nopentelemetry-semantic-conventions==0.66b2\noutlines==0.1.11\noutlines-core==0.1.26\npackaging==26.3\npartial-json-parser==0.2.1.1.post7\npeft==0.17.1\npillow==13.0.0\nprometheus-client==0.26.0\nprometheus-fastapi-instrumentator==8.1.0\npropcache==0.5.4\nprotobuf==7.36.2\npsutil==7.2.2\npy-cpuinfo==9.0.0\npybase64==1.5.1\npycountry==26.2.16\npydantic==2.14.2\npydantic-core==2.50.2\npydantic-extra-types==2.11.1\npydantic-settings==2.16.2\npygments==2.21.0\npython-dotenv==1.2.4\npython-json-logger==4.2.0\npython-multipart==0.0.33\npyyaml==6.0.3\npyzmq==27.2.0\nray==2.59.0\nreferencing==0.37.0\nregex==2026.9.29\nrequests==2.34.2\nrich==15.0.0\nrich-toolkit==0.20.6\nrignore==0.8.1\nrpds-py==2026.9.1\nsafetensors==0.8.0\nscipy==1.18.1\nsentencepiece==0.2.2\nsentry-sdk==2.73.0\nsetuptools==79.0.1\nshellingham==1.5.4\nsix==1.17.0\nsniffio==1.3.1\nstarlette==1.10.0\nsympy==1.14.0\ntiktoken==0.14.0\ntokenizers==0.21.4\ntorch==2.7.0+cu128\ntorchaudio==2.7.0+cu128\ntorchvision==0.22.0+cu128\ntqdm==4.70.1\ntransformers==4.53.3\ntriton==3.3.0\ntruststore==0.10.5\ntyper==0.27.3\ntyping-extensions==4.16.0\ntyping-inspection==0.4.4\nurllib3==2.8.0\nuvicorn==0.54.0\nuvloop==0.23.0\nvllm==0.9.2\nwatchfiles==1.3.0\nwebsockets==17.4\nwrapt==1.17.3\nxformers==0.0.30\nxgrammar==0.1.19\nyarl==1.25.1\n', 'split.txt': 'train_1474\ntrain_0255\ntrain_0063\ntrain_1715\ntrain_0618\ntrain_0538\ntrain_0486\ntrain_0313\ntrain_1703\ntrain_0236\ntrain_1560\ntrain_1713\ntrain_1249\ntrain_0203\ntrain_1356\ntrain_0959\ntrain_0080\ntrain_0075\ntrain_0216\ntrain_0476\ntrain_0508\ntrain_1155\ntrain_1379\ntrain_0066\ntrain_1289\ntrain_0435\ntrain_1653\ntrain_1497\ntrain_1617\ntrain_0953\ntrain_0480\ntrain_1021\ntrain_1353\ntrain_0624\ntrain_0015\ntrain_0354\ntrain_1608\ntrain_0960\ntrain_0779\ntrain_0346\ntrain_0469\ntrain_0772\ntrain_0214\ntrain_0864\ntrain_0225\ntrain_0819\ntrain_0787\ntrain_1384\ntrain_0596\ntrain_0106\ntrain_1623\ntrain_0923\ntrain_1300\ntrain_1486\ntrain_1352\ntrain_1355\ntrain_1014\ntrain_1380\ntrain_1542\ntrain_1295\ntrain_0943\ntrain_1070\ntrain_1705\ntrain_1448\ntrain_0958\ntrain_1081\ntrain_1137\ntrain_1248\ntrain_1478\ntrain_1305\ntrain_0017\ntrain_0977\ntrain_0933\ntrain_1016\ntrain_1128\ntrain_1460\ntrain_1124\ntrain_1658\ntrain_1265\ntrain_1491\ntrain_1432\ntrain_1278\ntrain_1405\ntrain_1546\ntrain_0867\ntrain_1117\ntrain_1654\ntrain_0505\ntrain_1118\ntrain_0572\ntrain_1148\ntrain_1433\ntrain_1556\ntrain_1557\ntrain_1615\ntrain_1317\ntrain_1215\ntrain_1346\ntrain_1440\ntrain_0536\n', 'umilife/infer_2gpu.py': '"""Fine-tuned multi-LoRA workflow; held-out labels are never model inputs."""\nimport os\nos.environ.setdefault("CUDA_VISIBLE_DEVICES","0")\nos.environ.setdefault("VLLM_USE_V1","0")\nos.environ.setdefault("VLLM_ATTENTION_BACKEND","XFORMERS")\nos.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD","spawn")\nos.environ.setdefault("VLLM_USAGE_STATS_ENABLED","0")\nos.environ.setdefault("DO_NOT_TRACK","1")\nimport json,time,argparse,math\nfrom pathlib import Path\nfrom collections import Counter\nfrom umilife.core import read_jsonl,write_json,dumps,parse_json,allowed_specs,load_catalog,import_toolenv\nfrom umilife.prompt_policy import (PROMPTS,COMPLETION_REVIEW,POLICY_AUDIT,POLICY_VERSION,\n    build_messages,request_schema,validate_meta_action,repeated_read,canonical,digest)\nfrom copy import deepcopy\nSYSTEMS=PROMPTS\nfrom umilife.data import normalized_row\nfrom umilife.grammar import role_schema\n\ndef rollout(row,text,specs,env,mode,max_calls=10,max_decisions=16):\n    started=time.monotonic();trajectory=[];trace=[];answer=None;reason="decision_limit"\n    feedback=None;failures=0;reviews=0;read_skips=0;repairs=0\n    def generate(role,plan=None,feedback=None):\n        schema=request_schema(role,specs,plan)\n        response=yield {"role":role,"messages":build_messages(role,row,text,specs,trajectory,plan,feedback),"schema":schema}\n        trace.append({"role":role,**response})\n        if response.get("schema_valid") is False:\n            raise ValueError("Model output violates request schema: "+response.get("schema_error",""))\n        return parse_json(response["raw"])\n    for _ in range(max_decisions):\n        try:\n            role="global" if mode=="single" else "planner"\n            planner_feedback=feedback\n            plan=yield from generate(role,feedback=planner_feedback);feedback=None\n            trace.append({"role":role,"output":plan})\n            if plan["next"] in ("summarizer","give_up"):\n                reviews+=1\n                trace.append({"event":"completion_review","proposal":plan})\n                plan=yield from generate(role,feedback={"kind":"completion_review",\'proposal\':plan,\'instruction\':COMPLETION_REVIEW})\n                trace.append({"role":role,"output":plan,"reviewed":True})\n            route=plan["next"]\n            if route=="give_up":reason="give_up";break\n            if route=="summarizer":\n                answer=plan.get("answer") if mode=="single" else (yield from generate("summarizer",plan)).get("answer")\n                reason="complete";break\n            if route!="caller" or plan.get("tool") not in specs:raise ValueError("Invalid planner route or unavailable tool")\n            if len(env.get_tool_calls())>=max_calls:reason="call_limit";break\n            action_feedback=planner_feedback;action=None\n            for attempt in range(3):\n                try:\n                    candidate=plan.get("action") if mode=="single" and attempt==0 else (\n                        (yield from generate(role,feedback=action_feedback)).get("action") if mode=="single" else\n                        (yield from generate("caller",plan,action_feedback)))\n                    if not isinstance(candidate,dict):raise ValueError("Invalid action object")\n                    name,args=candidate.get("name"),candidate.get("arguments")\n                    if name!=plan[\'tool\']:raise ValueError("Caller must use the tool chosen by Planner")\n                    check=env.validate(name,args) if name in specs and isinstance(args,dict) else {"ok":False,"error":"Unknown tool or invalid arguments"}\n                    if check.get(\'ok\'):\n                        extra=validate_meta_action(candidate,specs)\n                        if extra:check={\'ok\':False,\'error\':extra}\n                    trace.append({"role":"caller","output":candidate,"validation":check})\n                    if check.get(\'ok\'):action=candidate;break\n                    error=check.get(\'error\',\'Invalid arguments\')\n                except (ValueError,KeyError,TypeError) as exc:error=str(exc)\n                repairs+=1\n                action_feedback={"kind":"tool_contract_error","error":error,\n                    "instruction":"Sửa theo schema và dữ kiện đã có; nếu thiếu căn cứ thì cần lập kế hoạch tra cứu hoặc hỏi bổ sung."}\n            if action is None:\n                failures+=1;feedback=action_feedback\n                if failures>=3:reason="format_error";trace.append({"error":"tool contract repair exhausted"});break\n                continue\n            if repeated_read(action,specs,trajectory):\n                read_skips+=1\n                feedback={\'kind\':\'repeated_read\',\'action\':action,\n                    \'instruction\':\'Kết quả lời gọi này đã có trong trajectory và chưa có thao tác ghi làm thay đổi dữ liệu. Dùng kết quả đó; nếu rỗng hãy đổi bộ lọc theo schema hoặc hỏi thiếu thông tin, không lặp nguyên lời gọi.\'}\n                trace.append({\'event\':\'repeat_read_prevented\',\'action\':action});continue\n            name,args=action[\'name\'],action[\'arguments\']\n            observed=env.call(name,args);failures=0\n            trajectory.append({\'step\':len(trajectory),\'thought\':plan.get(\'thought\',\'\'),\'action\':action,\'observation\':observed})\n            if name==\'sys.ask_missing_info\':\n                terminal={\'thought\':\'Đã yêu cầu bổ sung thông tin; chờ người dùng trả lời.\',\'next\':\'summarizer\'}\n                answer=(yield from generate(\'summarizer\',terminal)).get(\'answer\')\n                reason=\'needs_user_input\';break\n        except (ValueError,KeyError,TypeError) as exc:\n            failures+=1;repairs+=1;trace.append({\'error\':str(exc)})\n            if failures>=3:reason=\'format_error\';break\n            feedback={\'kind\':\'planning_contract_error\',\'error\':str(exc),\'instruction\':\'Lập lại đúng một bước theo dữ kiện và schema công cụ đã cài.\'}\n    return {\'id\':row[\'id\'],\'tool_calls\':env.get_tool_calls(),\'state_hash\':env.get_state_hash()},{\n        \'id\':row[\'id\'],\'reason\':reason,\'elapsed_s\':time.monotonic()-started,\n        \'over_deadline\':False,\'time_limit_enforced\':False,\'answer\':answer,\n        \'trace\':trace,\'trajectory\':trajectory,\'transcript\':text,\n        \'completion_reviews\':reviews,\'repeat_reads_prevented\':read_skips,\'repairs\':repairs,\'prompt_policy\':POLICY_VERSION}\n\ndef run_batch(generators,backend):\n    active={}\n    completed={}\n    for index,generator in enumerate(generators):\n        try:active[index]=next(generator)\n        except StopIteration as stop:completed[index]=stop.value\n    while active:\n        good=[];bad=[]\n        for index,request in active.items():\n            try:backend.validate(request);good.append(index)\n            except ValueError as exc:bad.append((index,exc))\n        for index,error in bad:\n            try:active[index]=generators[index].throw(error)\n            except StopIteration as stop:completed[index]=stop.value;del active[index]\n        if not good:continue\n        responses=backend.generate([active[index] for index in good])\n        assert len(responses)==len(good)\n        for index,response in zip(good,responses,strict=True):\n            try:active[index]=generators[index].send(response)\n            except StopIteration as stop:completed[index]=stop.value;del active[index]\n    return [completed[index] for index in range(len(generators))]\n\ndef prepare_t4_lora_kernel():\n    """Work around Triton 3.3 fp16 conversion failure on SM75; preserve FP32 accumulation."""\n    import torch,importlib.util,importlib.metadata,hashlib\n    path=Path(importlib.util.find_spec("vllm").origin).parent/"lora/ops/triton_ops/kernel_utils.py"\n    original="        accumulator += tl.dot(\\n            tiled_a,\\n            tiled_b,\\n        )"\n    patched="        # LifeMate SM75 compatibility: explicit exact fp16->fp32 before dot.\\n        accumulator += tl.dot(tiled_a.to(tl.float32), tiled_b.to(tl.float32), input_precision=\\"ieee\\")"\n    source=path.read_text()\n    capability=torch.cuda.get_device_capability(0)\n    if capability[0]>=8:\n        if patched in source:path.write_text(source.replace(patched,original))\n        return {"applied":False,"capability":list(capability)}\n    assert importlib.metadata.version("vllm")=="0.9.2"\n    assert importlib.metadata.version("triton")=="3.3.0"\n    assert original in source or patched in source,"Unexpected LoRA kernel source"\n    if original in source:path.write_text(source.replace(original,patched))\n    # Exercise the exact patched helper on the GPU against an FP32 PyTorch reference.\n    global triton,tl,mm_k\n    import triton\n    import triton.language as tl\n    from vllm.lora.ops.triton_ops.kernel_utils import mm_k\n    @triton.jit\n    def check_kernel(A,B,C,K:tl.constexpr):\n        rows=tl.arange(0,32)\n        cols=tl.arange(0,16)\n        kk=tl.arange(0,32)\n        a=A+rows[:,None]*K+kk[None,:]\n        b=B+kk[:,None]*16+cols[None,:]\n        acc=mm_k(a,b,1,16,kk,K,32,16,32,True,1,False,tl.float16)\n        tl.store(C+rows[:,None]*16+cols[None,:],acc)\n    errors=[]\n    torch.manual_seed(42)\n    for k in (128,2560):\n        a=torch.randn(32,k,device="cuda",dtype=torch.float16)\n        b=torch.randn(k,16,device="cuda",dtype=torch.float16)\n        c=torch.empty(32,16,device="cuda",dtype=torch.float32)\n        check_kernel[(1,)](a,b,c,k,num_warps=4)\n        ref=a.float()@b.float()\n        torch.testing.assert_close(c,ref,rtol=5e-4,atol=2e-3)\n        errors.append(float((c-ref).abs().max()))\n    return {"applied":True,"capability":list(capability),"operation":"LoRA dot with FP32 accumulation",\n            "reference_test":"pass","maximum_errors":errors,"kernel_sha256":hashlib.sha256(path.read_bytes()).hexdigest()}\n\nclass Backend:\n    def __init__(self,model,revision,adapters,max_context=8192):\n        self.t4_compatibility=prepare_t4_lora_kernel()\n        print(dumps({"t4_compatibility":self.t4_compatibility}),flush=True)\n        from vllm import LLM,SamplingParams\n        from vllm.sampling_params import GuidedDecodingParams\n        from vllm.lora.request import LoRARequest\n        from transformers import AutoConfig,AutoModelForCausalLM\n        from safetensors import safe_open\n        import torch\n        assert torch.cuda.is_available()\n        assert torch.cuda.get_device_capability(0)>=(7,0)\n        self.max_context=max_context\n        self.requests={}\n        adapter_parameters=0\n        lineage_records={}\n        for index,role in enumerate(("global","planner"),start=1):\n            path=Path(adapters)/role\n            meta=json.loads((path/"lineage.json").read_text())\n            assert meta["stage"]==role and meta["model"]==model and meta["revision"]==revision\n            assert meta["final_adapter"]["finite"]\n            assert meta[\'prompt_sha256\']==digest(PROMPTS),\'Training/inference policy mismatch\'\n            assert meta[\'world_size\']==2 and meta[\'micro_batch\']>0 and meta[\'eval_batch_per_device\']>0\n            lineage_records[role]=meta\n            self.requests[role]=LoRARequest(role,index,str(path))\n            with safe_open(path/"adapter_model.safetensors",framework="pt",device="cpu") as handle:\n                adapter_parameters+=sum(math.prod(handle.get_slice(k).get_shape()) for k in handle.keys())\n        manifest_hashes={v[\'dataset_manifest_sha256\'] for v in lineage_records.values()}\n        assert len(manifest_hashes)==1\n        for role in (\'planner\',):\n            assert lineage_records[role][\'initial_adapter\'][\'sha256\']==lineage_records[\'global\'][\'final_adapter\'][\'sha256\']\n        # Missing local roles use the completed global adapter explicitly; no fake local checkpoints.\n        self.requests[\'caller\']=self.requests[\'global\'];self.requests[\'summarizer\']=self.requests[\'global\']\n        self.dataset_manifest_sha256=next(iter(manifest_hashes))\n        config=AutoConfig.from_pretrained(model,revision=revision)\n        with torch.device("meta"):meta_model=AutoModelForCausalLM.from_config(config)\n        base_parameters=meta_model.num_parameters();del meta_model\n        self.parameter_report={"base":base_parameters,"comparison_adapters":adapter_parameters,\n                               "total_llm":base_parameters+adapter_parameters}\n        assert self.parameter_report["total_llm"]<5_000_000_000\n        self.llm=LLM(model=model,revision=revision,tokenizer=str(Path(adapters)/"global"),\n            dtype="float16",tensor_parallel_size=1,enable_lora=True,max_loras=4,max_cpu_loras=4,\n            max_lora_rank=16,max_model_len=max_context,max_num_seqs=4,max_num_batched_tokens=max_context,\n            gpu_memory_utilization=.90,enforce_eager=True,enable_prefix_caching=False,\n            enable_chunked_prefill=False,seed=42,trust_remote_code=False,\n            generation_config="vllm",disable_log_stats=True)\n        self.tokenizer=self.llm.get_tokenizer()\n        self.max_tokens={"global":768,"planner":320,"caller":512,"summarizer":384}\n        self.batch_sizes=[];self.roles_seen=Counter();self.schema_violations=0\n    def make_sampling(self,request):\n        from vllm import SamplingParams\n        from vllm.sampling_params import GuidedDecodingParams\n        # Fresh per request: never share mutable grammar state between batch members.\n        return SamplingParams(temperature=0,seed=42,\n            max_tokens=64 if request.get("probe") else self.max_tokens[request["role"]],\n            guided_decoding=GuidedDecodingParams(\n                json=request.get("schema",role_schema(request["role"])),backend="lm-format-enforcer"))\n    def validate(self,request):\n        role=request["role"]\n        prompt=self.tokenizer.apply_chat_template(request["messages"],tokenize=False,add_generation_prompt=True)\n        length=len(self.tokenizer.encode(prompt,add_special_tokens=False))\n        if length+(64 if request.get("probe") else self.max_tokens[role])>self.max_context:\n            raise ValueError("Context budget exceeded; no evidence is truncated")\n    def generate(self,requests):\n        assert 1<=len(requests)<=4\n        for r in requests:self.validate(r)\n        self.batch_sizes.append(len(requests));self.roles_seen.update(r["role"] for r in requests)\n        prompts=[self.tokenizer.apply_chat_template(r["messages"],tokenize=False,add_generation_prompt=True) for r in requests]\n        results=self.llm.generate(prompts,[self.make_sampling(r) for r in requests],\n            lora_request=[self.requests[r["role"]] for r in requests],use_tqdm=False)\n        from jsonschema import Draft202012Validator,ValidationError\n        responses=[]\n        for request,result in zip(requests,results,strict=True):\n            response={"raw":result.outputs[0].text,"finish_reason":result.outputs[0].finish_reason,\n                      "stop_reason":result.outputs[0].stop_reason,\n                      "prompt_sha256":digest(request["messages"]),"schema_sha256":digest(request.get("schema",role_schema(request["role"])))}\n            try:\n                value=parse_json(response["raw"])\n                Draft202012Validator(request.get("schema",role_schema(request["role"]))).validate(value)\n                response["schema_valid"]=True\n            except (ValueError,ValidationError) as exc:\n                response.update(schema_valid=False,schema_error=str(exc)[:300])\n                self.schema_violations+=1\n            responses.append(response)\n        return responses\n    def preflight(self):\n        probes=[]\n        for role in ("global","planner","caller","summarizer"):\n            schema={"type":"object","properties":{"probe":{"type":"string","enum":[role]}},\n                    "required":["probe"],"additionalProperties":False}\n            probes.append({"role":role,"probe":True,"schema":schema,"messages":[\n                {"role":"user","content":"Just say hello, no JSON."}]})\n        for request,result in zip(probes,self.generate(probes),strict=True):\n            assert result["schema_valid"] and parse_json(result["raw"])=={"probe":request["role"]},result\n        specs={"demo.inspect":{"name":"demo.inspect","kind":"read","description":"Read a synthetic probe record",\n            "parameters":{"type":"object","properties":{"token":{"type":"string","enum":["probe"]}},"required":["token"]}}}\n        row={"context":{},"history":[]};plan={"thought":"Cần đọc dữ liệu kiểm tra.","next":"caller","tool":"demo.inspect"}\n        requests=[]\n        for role in ("global","planner","caller","summarizer"):\n            trajectory=[] if role!=\'summarizer\' else [{\'action\':{\'name\':\'demo.inspect\',\'arguments\':{\'token\':\'probe\'}},\'observation\':{\'ok\':True,\'value\':\'probe\'}}]\n            requests.append({"role":role,"messages":build_messages(role,row,\'Đọc bản ghi kiểm tra được cung cấp.\',specs,trajectory,plan),\n                             \'schema\':request_schema(role,specs,plan)})\n        results=self.generate(requests)\n        assert all(r[\'schema_valid\'] for r in results),results\n        assert parse_json(results[2][\'raw\'])=={\'name\':\'demo.inspect\',\'arguments\':{\'token\':\'probe\'}}\n        contracts=[]\n        for name,parameters in [\n            (\'demo.integer\',{\'type\':\'object\',\'properties\':{\'count\':{\'type\':\'integer\',\'enum\':[2,4]}},\'required\':[\'count\']}),\n            (\'demo.alternative\',{\'type\':\'object\',\'properties\':{\'reference\':{\'type\':\'string\'},\'code\':{\'type\':\'integer\'}},\'required\':[],\'x-one-of\':[[\'reference\'],[\'code\']]}),\n            (\'demo.optional\',{\'type\':\'object\',\'properties\':{\'choice\':{\'type\':\'string\',\'enum\':[\'alpha\',\'beta\']}},\'required\':[]}),\n            (\'demo.flag\',{\'type\':\'object\',\'properties\':{\'enabled\':{\'type\':\'boolean\'}},\'required\':[\'enabled\']})]:\n            item={name:{\'name\':name,\'kind\':\'read\',\'description\':\'Synthetic tool contract probe\',\'parameters\':parameters}}\n            probe_plan={\'thought\':\'Kiểm tra schema giả lập.\',\'next\':\'caller\',\'tool\':name}\n            contracts.append({\'role\':\'caller\',\'schema\':request_schema(\'caller\',item,probe_plan),\n                \'messages\':build_messages(\'caller\',row,\'Thực hiện kiểm tra schema công cụ giả lập.\',item,[],probe_plan)})\n        results=self.generate(contracts)\n        assert all(r[\'schema_valid\'] for r in results),results\n        return {\'real_vllm_batch_size\':4,\'tested_prompt_roles\':list(self.requests),\'json_contracts\':\'pass\',\n            \'adversarial_schema_probe\':\'pass\',\'selected_tool_and_argument_schema_probes\':\'pass\',\n            \'probe_source\':\'synthetic fixtures only; no benchmark examples\'}\n\ndef main():\n    p=argparse.ArgumentParser()\n    for name in (\'data-root\',\'work\',\'model\',\'revision\',\'adapters\',\'ids-file\'):p.add_argument(\'--\'+name,required=True)\n    p.add_argument(\'--deadline-monotonic\',type=float)\n    p.add_argument(\'--split\',choices=[\'validation\',\'public_test\'],required=True)\n    p.add_argument(\'--mode\',choices=[\'single\',\'multi_fallback\'],default=\'multi_fallback\')\n    a=p.parse_args();root=Path(a.data_root);work=Path(a.work);work.mkdir(parents=True,exist_ok=True)\n    split=\'train\' if a.split==\'validation\' else \'public_test\'\n    ids=Path(a.ids_file).read_text().splitlines();assert len(ids)==len(set(ids)) and ids\n    source=read_jsonl(root/split/f\'{split}.jsonl\');lookup={r[\'id\']:r for r in source};assert set(ids)<=lookup.keys()\n    if a.split==\'public_test\':assert set(ids)==set(lookup),\'Test must include every public-test ID\'\n    rows=[]\n    for sid in ids:\n        raw=lookup[sid]\n        context={k:deepcopy(raw[\'context\'][k]) for k in (\'current_time\',\'timezone\',\'location\',\'user_profile\') if k in raw[\'context\']}\n        context[\'installed_apps\']={app:{\'tools\':list(meta.get(\'tools\',[]))} for app,meta in raw[\'context\'][\'installed_apps\'].items()}\n        history=[]\n        for turn in raw.get(\'history\',[]):\n            keys=(\'role\',\'audio\') if turn[\'role\']==\'user\' else (\'role\',\'text\',\'tool_calls\',\'tool_results\')\n            history.append({k:deepcopy(turn[k]) for k in keys if k in turn})\n        rows.append({\'id\':sid,\'audio\':raw[\'audio\'],\'context\':context,\'history\':history})\n    del source,lookup\n    transcripts=json.loads((work/\'asr_transcripts.json\').read_text());asr=json.loads((work/\'asr_report.json\').read_text())\n    assert not asr[\'oracle_fallback\'] and asr[\'validation_samples\']==len(rows)\n    backend=Backend(a.model,a.revision,a.adapters);preflight=backend.preflight();write_json(work/\'preflight.json\',preflight)\n    catalog=load_catalog(root,split);ToolEnv=import_toolenv(root,split)\n    batches=work/\'batches\';batches.mkdir(exist_ok=True)\n    active=work/\'active_batch.json\'\n    if active.exists():\n        marker=json.loads(active.read_text())\n        assert (batches/marker[\'file\']).exists(),\'An interrupted batch has unrecorded tool execution. Do not silently replay it.\'\n        active.unlink()\n    predictions=[];traces=[];mode=\'single\' if a.mode==\'single\' else \'multi\'\n    with (work/\'submission.jsonl\').open(\'w\') as pf,(work/\'traces.jsonl\').open(\'w\') as tf:\n        for start in range(0,len(rows),4):\n            if a.deadline_monotonic and time.monotonic()>a.deadline_monotonic and not (batches/f\'{start:06}.json\').exists():\n                write_json(work/\'progress.json\',{\'paused\':True,\'completed\':len(predictions),\'total\':len(rows),\'next_index\':start})\n                print(\'PAUSED_AT_BATCH_BOUNDARY: preserve this run directory and resume in another session\',flush=True)\n                raise SystemExit(75)\n            group=rows[start:start+4]\n            batch_file=batches/f\'{start:06}.json\'\n            if batch_file.exists():\n                completed=json.loads(batch_file.read_text())\n                assert [p[0][\'id\'] for p in completed]==[r[\'id\'] for r in group]\n            else:\n                write_json(active,{\'file\':batch_file.name,\'ids\':[r[\'id\'] for r in group]})\n            if not batch_file.exists():\n                envs=[ToolEnv(split,raw[\'id\'],root=root) for raw in group]\n                safe_rows=[normalized_row(raw,transcript=transcripts) for raw in group]\n                gens=[rollout(row,transcripts[raw[\'audio\']],allowed_specs(row,catalog),env,mode)\n                      for raw,row,env in zip(group,safe_rows,envs,strict=True)]\n                completed=run_batch(gens,backend)\n                temp=batch_file.with_suffix(\'.tmp\');write_json(temp,completed);temp.replace(batch_file);active.unlink()\n            for pred,trace in completed:\n                predictions.append(pred);traces.append(trace);pf.write(dumps(pred)+\'\\n\');pf.flush();tf.write(dumps(trace)+\'\\n\');tf.flush()\n            print(dumps({\'split\':a.split,\'mode\':a.mode,\'completed\':len(predictions),\'total\':len(rows)}),flush=True)\n    assert [r[\'id\'] for r in predictions]==ids\n    report={\'split\':a.split,\'mode\':a.mode,\'samples\':len(rows),\'asr\':asr,\'preflight\':preflight,\n            \'actual_batch_sizes\':dict(Counter(backend.batch_sizes)),\'schema_violations\':backend.schema_violations,\n            \'role_adapter_map\':{\'global\':\'global\',\'planner\':\'planner\',\'caller\':\'global\',\'summarizer\':\'global\'},\n            \'missing_local_adapters\':[\'caller\',\'summarizer\'],\'policy\':POLICY_AUDIT,\n            \'termination_reasons\':dict(Counter(t[\'reason\'] for t in traces))}\n    if a.split==\'validation\':\n        # Scorer-only gold replay after ALL model predictions have been generated.\n        from umilife.evaluate import evaluate\n        references=[]\n        for raw in read_jsonl(root/\'train/train.jsonl\'):\n            if raw[\'id\'] not in set(ids):continue\n            env=ToolEnv(\'train\',raw[\'id\'],root=root)\n            for action in raw[\'label\'][\'tool_calls\']:env.call(action[\'name\'],action[\'arguments\'])\n            references.append({\'id\':raw[\'id\'],\'tool_calls\':raw[\'label\'][\'tool_calls\'],\'state_hash\':env.get_state_hash()})\n        report[\'metrics\']=evaluate(predictions,references,catalog,traces)\n    write_json(work/\'report.json\',report)\n    print(dumps({k:v for k,v in report.items() if k!=\'metrics\'}),flush=True)\nif __name__==\'__main__\':main()\n', 'umilife/prompt_policy.py': '"""Generic tool policy. No benchmark examples, held-out labels or per-sample rules."""\nfrom copy import deepcopy\nfrom datetime import datetime,timedelta\nimport hashlib,json\n\nPOLICY_VERSION=\'generic-schema-and-completion-v2.1\'\nCOMMON=\'\'\'Bạn là trợ lý LifeMate tiếng Việt, thực hiện yêu cầu bằng công cụ đã cài.\nChỉ xử lý request hiện tại. History là các lượt trước: dùng để hiểu tham chiếu và lấy thông tin đã biết, nhưng không coi hành động trong history là đã hoàn thành request mới. Chỉ trajectory hiện tại chứng minh các bước vừa thực hiện.\nRequest nguyên văn là nguồn chính: giữ đủ từ hạn định đối tượng, số lượng, đơn vị và phủ định; không thay bằng một bản diễn giải mất chi tiết trong thought.\nĐọc toàn bộ ý định, lời sửa đổi, điều kiện và nhiều việc trong cùng câu. Văn bản đến từ nhận dạng tiếng nói: diễn giải theo ngữ cảnh, đơn vị và danh mục công cụ; không tự bịa tên riêng, ID hoặc dữ kiện.\nƯu tiên thông tin có sẵn trong context, history, trajectory. Khi thiếu ID hay dữ liệu, dùng công cụ tra cứu phù hợp; x-ref mô tả nguồn định danh. Không chọn ngẫu nhiên khi kết quả có nhiều đối tượng. Nếu tham số bắt buộc không thể tra cứu hoặc suy ra chắc chắn thì hỏi qua công cụ hệ thống báo thiếu thông tin.\nHiểu số và ngày giờ từ request, dùng current_time và timezone của context. Giờ tương đối phải tính đúng ngày; biểu thức giờ kém phút là giờ tròn trừ số phút, sau khi xác định buổi. Giữ đơn vị và kiểm tra phép tính trước khi tạo arguments.\nVới điều kiện, đọc dữ liệu để xác định nhánh rồi mới thực hiện thao tác phụ thuộc. Không thêm bước đọc khi thông tin cần thiết đã có. Tránh lặp cùng lời gọi khi chưa có dữ kiện mới, tránh thực hiện lại thao tác ghi đã thành công nếu người dùng không yêu cầu lặp.\nMột observation có ok=true nhưng done=false vẫn chưa thực hiện được thao tác. Kết quả tra cứu rỗng không chứng minh yêu cầu ngoài phạm vi: cân nhắc điều chỉnh bộ lọc, tra cứu rộng hơn theo schema hoặc hỏi thông tin còn thiếu. Không tưởng tượng kết quả.\nChỉ xuất một JSON đúng hợp đồng đầu ra, không markdown, không văn bản ngoài JSON.\'\'\'\n\nPROMPTS={\n \'planner\':COMMON+\'\'\'\nBạn là Planner. Chọn đúng một bước tiếp theo theo ý định và những gì thực sự còn thiếu.\nThought chỉ một hoặc hai câu ngắn về bước còn thiếu, bằng chứng đã có và lý do chọn bước này; không kể lại toàn bộ request. Khi next=caller, tool phải là tên chính xác có trong tools. Xem kind, parameters, required và x-ref để biết cách lấy dữ liệu phụ thuộc trước khi ghi.\nChỉ chọn summarizer khi mọi ý của request đã được thực hiện có bằng chứng, điều kiện không cần thực hiện đã được kiểm tra, người dùng không yêu cầu thao tác, hoặc đã gọi công cụ hỏi bổ sung và cần chờ trả lời. Tìm được thông tin chưa đồng nghĩa đã thực hiện hành động người dùng muốn.\nChỉ chọn give_up khi yêu cầu ngoài khả năng của toàn bộ công cụ đã cài. Thiếu một ID, tra cứu rỗng hoặc chưa biết cách dùng tool không tự động có nghĩa ngoài phạm vi.\nJSON gồm thought và next. next là caller, summarizer hoặc give_up. Chỉ khi next=caller mới thêm tool.\'\'\',\n \'caller\':COMMON+\'\'\'\nBạn là Caller. Tạo đúng một lời gọi cho tool Planner đã chọn. Output có đúng hai trường name và arguments; name phải khớp tên trong selected_tool, arguments tuân thủ schema được cung cấp.\nĐọc kỹ ý định, plan và observation để điền giá trị. Dùng đúng kiểu số nguyên/số thực/chuỗi/mảng và đúng enum. Không bọc lời gọi vào một tool khác. Không tạo trường ngoài properties. Tham số tùy chọn không có căn cứ thì bỏ qua; không thêm bộ lọc phỏng đoán. Nhóm x-one-of chỉ cần đủ một phương án có dữ kiện.\nPlan chỉ chọn bước và tool. Luôn đối chiếu giá trị và đối tượng với request nguyên văn; không sao chép một số hay tên sai từ thought. Giữ đủ từ hạn định của tên cần tìm khi dùng bộ lọc.\nVới nội dung gửi hoặc lưu, giữ đúng nội dung người dùng yêu cầu, không tự viết thêm lời bình luận. Không khẳng định đã gọi công cụ: chương trình sẽ thực hiện sau khi bạn trả JSON.\'\'\',\n \'summarizer\':COMMON+\'\'\'\nBạn là Summarizer. Viết answer ngắn, tự nhiên và trung thực từ observation của trajectory hiện tại. Không gọi thêm tool và không chép toàn bộ JSON kết quả.\nChỉ báo một thao tác thành công nếu đúng tool đã thực sự chạy và observation xác nhận. Nếu chưa thực hiện thì nói rõ; nếu đã hỏi bổ sung thì nêu thông tin còn cần và chờ người dùng. Không lấy lời nói của assistant trong history làm bằng chứng thực hiện request mới.\nOutput chỉ có answer.\'\'\',\n \'global\':COMMON+\'\'\'\nBạn xử lý từng bước theo ReAct. Output thought và next. next=caller cần tool và action gồm name, arguments đúng schema; next=summarizer cần answer ngắn dựa trên kết quả thực tế; next=give_up chỉ khi ngoài khả năng toàn bộ tool đã cài.\nĐối chiếu toàn bộ mục tiêu với trajectory trước khi kết thúc. Dữ liệu đã tìm thấy chưa có nghĩa thao tác yêu cầu đã thực hiện.\'\'\'\n}\n\nCOMPLETION_REVIEW=\'\'\'Kiểm tra lại đề xuất kết thúc trước khi trả lời. Đây chưa phải thao tác thực thi.\nĐối chiếu TỪNG ý của request với trajectory hiện tại. Có việc chỉ mới được tra thông tin nhưng chưa thực hiện không? Có đang nhầm history với request hiện tại không? Có observation done=false hoặc kết quả rỗng chưa được xử lý không?\nNếu còn việc có thể thực hiện, chọn caller và tool kế tiếp. Nếu thiếu thông tin thì ưu tiên tra cứu, chỉ hỏi bổ sung khi không thể tìm. Chỉ summarizer khi đã đủ bằng chứng hoàn thành/không cần thao tác/đang chờ người dùng; give_up chỉ khi ngoài khả năng tool đã cài.\'\'\'\n\ndef canonical(value):return json.dumps(value,ensure_ascii=False,sort_keys=True,separators=(\',\',\':\'))\ndef digest(value):return hashlib.sha256(canonical(value).encode()).hexdigest()\ndef obj(properties,required):return {\'type\':\'object\',\'properties\':properties,\'required\':required,\'additionalProperties\':False}\n\ndef parameter_schema(schema):\n    """Convert documented x-one-of to ordinary JSON Schema; preserve typed nesting."""\n    allowed={\'type\',\'properties\',\'required\',\'items\',\'enum\',\'format\',\'minimum\',\'maximum\',\n             \'exclusiveMinimum\',\'exclusiveMaximum\',\'minLength\',\'maxLength\',\'minItems\',\'maxItems\',\n             \'anyOf\',\'oneOf\',\'allOf\',\'additionalProperties\',\'description\'}\n    out={}\n    for key,value in schema.items():\n        if key not in allowed:continue\n        if key==\'properties\':out[key]={name:parameter_schema(prop) for name,prop in value.items()}\n        elif key==\'items\' and isinstance(value,dict):out[key]=parameter_schema(value)\n        elif key in (\'anyOf\',\'oneOf\',\'allOf\'):out[key]=[parameter_schema(x) for x in value]\n        else:out[key]=deepcopy(value)\n    if out.get(\'type\')==\'object\' and \'properties\' in out:out[\'additionalProperties\']=False\n    if schema.get(\'x-one-of\'):\n        # Full object alternatives are supported by both JSON Schema and LMFE.\n        base=out\n        out={\'anyOf\':[{**deepcopy(base),\'required\':list(dict.fromkeys(base.get(\'required\',[])+list(group)))}\n                      for group in schema[\'x-one-of\']]}\n    return out\n\ndef action_schema(specs,name):\n    arguments=parameter_schema(specs[name][\'parameters\'])\n    if name==\'sys.ask_missing_info\':\n        # Only public installed schemas provide these choices, never an answer label.\n        choices=[n for n,s in specs.items() if n!=name and s[\'parameters\'].get(\'properties\')]\n        if choices:arguments[\'properties\'][\'tool_name\'][\'enum\']=sorted(choices)\n        arguments[\'properties\'][\'params\'][\'minItems\']=1\n        fields=sorted({key for target in choices for key in specs[target][\'parameters\'][\'properties\']})\n        if fields:arguments[\'properties\'][\'params\'][\'items\'][\'enum\']=fields\n    return obj({\'name\':{\'type\':\'string\',\'enum\':[name]},\'arguments\':arguments},[\'name\',\'arguments\'])\n\ndef route_schema(specs,global_mode=False):\n    thought={\'type\':\'string\',\'maxLength\':480 if global_mode else 240};branches=[]\n    if specs:\n        if global_mode:\n            for name in sorted(specs):\n                branches.append(obj({\'thought\':thought,\'next\':{\'type\':\'string\',\'enum\':[\'caller\']},\n                    \'tool\':{\'type\':\'string\',\'enum\':[name]},\'action\':action_schema(specs,name)},[\'thought\',\'next\',\'tool\',\'action\']))\n        else:\n            branches.append(obj({\'thought\':thought,\'next\':{\'type\':\'string\',\'enum\':[\'caller\']},\n                \'tool\':{\'type\':\'string\',\'enum\':sorted(specs)}},[\'thought\',\'next\',\'tool\']))\n    done={\'thought\':thought,\'next\':{\'type\':\'string\',\'enum\':[\'summarizer\']}}\n    if global_mode:done[\'answer\']={\'type\':\'string\',\'maxLength\':1024}\n    branches.append(obj(done,list(done)))\n    branches.append(obj({\'thought\':thought,\'next\':{\'type\':\'string\',\'enum\':[\'give_up\']}},[\'thought\',\'next\']))\n    return {\'anyOf\':branches}\n\ndef compact_catalog(specs):\n    fields=(\'type\',\'enum\',\'x-ref\',\'format\',\'items\',\'minimum\',\'maximum\')\n    return [{\'name\':name,\'kind\':spec.get(\'kind\'),\'description\':spec[\'description\'],\n             \'parameters\':{key:{k:deepcopy(v) for k,v in prop.items() if k in fields}\n                           for key,prop in spec[\'parameters\'].get(\'properties\',{}).items()},\n             \'required\':spec[\'parameters\'].get(\'required\',[]),\n             **({\'one_of\':spec[\'parameters\'][\'x-one-of\']} if spec[\'parameters\'].get(\'x-one-of\') else {})}\n            for name,spec in sorted(specs.items())]\n\ndef reference_clock(context):\n    result={}\n    try:\n        now=datetime.fromisoformat(context[\'current_time\'])\n        result={\'today\':now.date().isoformat(),\'tomorrow\':(now+timedelta(days=1)).date().isoformat(),\n                \'day_after_tomorrow\':(now+timedelta(days=2)).date().isoformat(),\n                \'weekday_iso\':now.isoweekday(),\'timezone\':context.get(\'timezone\')}\n    except (KeyError,TypeError,ValueError):pass\n    return result\n\ndef build_messages(role,row,text,specs,trajectory,plan=None,feedback=None):\n    # Explicit provenance allowlist. Current labels/final state/reference transcript never enter here.\n    context={k:deepcopy(row[\'context\'][k]) for k in (\'current_time\',\'timezone\',\'location\',\'user_profile\') if k in row[\'context\']}\n    history=[{k:deepcopy(turn[k]) for k in (\'role\',\'text\',\'tool_calls\',\'tool_results\') if k in turn}\n             for turn in row.get(\'history\',[])]\n    content={\'context\':context,\'reference_clock\':reference_clock(context),\'history\':history,\n             \'request\':text,\'trajectory\':deepcopy(trajectory)}\n    if role in (\'planner\',\'global\'):content[\'tools\']=compact_catalog(specs)\n    elif role==\'caller\':\n        selected=(plan or {}).get(\'tool\');assert selected in specs,\'Planner selected an unavailable tool\'\n        content.update(plan=deepcopy(plan),selected_tool=deepcopy(specs[selected]))\n        if selected==\'sys.ask_missing_info\':\n            content[\'missing_information_tools\']=compact_catalog({name:spec for name,spec in specs.items()\n                if name!=selected and spec[\'parameters\'].get(\'properties\')})\n    else:content[\'plan\']=deepcopy(plan)\n    if feedback is not None:content[\'validation_feedback\']=deepcopy(feedback)\n    # Keep the original request at the end, after schemas/observations/feedback.\n    content.pop(\'request\');content[\'request\']=text\n    return [{\'role\':\'system\',\'content\':PROMPTS[role]},\n            {\'role\':\'user\',\'content\':json.dumps(content,ensure_ascii=False,separators=(\',\',\':\'))}]\n\ndef request_schema(role,specs,plan=None):\n    if role==\'caller\':return action_schema(specs,plan[\'tool\'])\n    if role in (\'planner\',\'global\'):return route_schema(specs,role==\'global\')\n    return obj({\'answer\':{\'type\':\'string\',\'maxLength\':1024}},[\'answer\'])\n\ndef validate_meta_action(action,specs):\n    if action[\'name\']!=\'sys.ask_missing_info\':return None\n    args=action[\'arguments\'];target=args.get(\'tool_name\');params=args.get(\'params\',[])\n    if target not in specs or target==\'sys.ask_missing_info\':return \'Tool cần bổ sung không có trong danh mục đã cài\'\n    # An optional API parameter can still be necessary to express the user\'s intent.\n    fields=set(specs[target][\'parameters\'].get(\'properties\',{}))\n    if not params or len(params)!=len(set(params)) or not set(params)<=fields:\n        return \'params phải là tên tham số có trong schema của tool_name, không phải câu hỏi tự do\'\n    return None\n\ndef repeated_read(action,specs,trajectory):\n    if specs[action[\'name\']].get(\'kind\')!=\'read\':return False\n    for old in reversed(trajectory):\n        if specs.get(old[\'action\'][\'name\'],{}).get(\'kind\')==\'write\':return False\n        if canonical(old[\'action\'])==canonical(action):return True\n    return False\n\nPOLICY_AUDIT={\'policy_version\':POLICY_VERSION,\'static_prompts_sha256\':digest(PROMPTS),\n    \'completion_review_sha256\':digest(COMPLETION_REVIEW),\'few_shot_examples\':0,\n    \'sample_specific_rules\':False,\'gold_inputs_to_model\':False,\n    \'allowed_input_sources\':[\'current ASR request\',\'ASR user history\',\'provided context\',\n        \'installed public tool schemas\',\'observations from executed tools\',\'model-generated plan/feedback\']}\n', 'umilife/transcribe_supplied.py': '"""Recognize current/history audio using only the supplied Zipformer2 ONNX files."""\nimport argparse,hashlib,json,math,time\nfrom pathlib import Path\nimport numpy as np\nimport soundfile as sf\nfrom scipy.signal import resample_poly\nimport sherpa_onnx\n\ndef digest(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\ndef transcribe(root,work,model,limit=None,threads=4,split="train"):\n    root=Path(root);work=Path(work);model=Path(model)\n    manifest=json.loads((model/\'model_manifest.json\').read_text())\n    assert manifest[\'model_type\']==\'zipformer2\' and not manifest[\'streaming\']\n    for name,info in manifest[\'files\'].items():assert digest(model/name)==info[\'sha256\'],f\'Wrong ASR file: {name}\'\n    fingerprint=digest(model/\'model_manifest.json\')\n    valid=set((work/\'split.txt\').read_text().splitlines());assert len(valid)>0\n    rows=[json.loads(line) for line in (root/split/f\'{split}.jsonl\').open()]\n    selected=[r for r in rows if r[\'id\'] in valid];assert len(selected)==len(valid)\n    if limit:selected=selected[:limit]\n    audio=[]\n    for row in selected:\n        audio.append(row[\'audio\'])\n        audio.extend(t[\'audio\'] for t in row.get(\'history\',[]) if t[\'role\']==\'user\')\n    audio=list(dict.fromkeys(audio))\n    config={\'model_manifest_sha256\':fingerprint,\'split_sha256\':digest(work/\'split.txt\'),\n        \'split\':split,\'selected_ids\':[r[\'id\'] for r in selected],\'sample_rate\':16000,\'decoding_method\':\'greedy_search\',\n        \'provider\':\'cpu\',\'sherpa_onnx\':sherpa_onnx.__version__,\'source\':\'supplied_asr_model\'}\n    config_path=work/\'asr_config.json\'\n    if config_path.exists():assert json.loads(config_path.read_text())==config,\'Stale ASR cache configuration\'\n    else:config_path.write_text(json.dumps(config,indent=2))\n    cache_path=work/\'asr_transcripts.jsonl\';cache={}\n    if cache_path.exists():\n        for line in cache_path.read_text().splitlines():\n            record=json.loads(line);assert record[\'model_manifest_sha256\']==fingerprint\n            cache[record[\'audio\']]=record\n    recognizer=sherpa_onnx.OfflineRecognizer.from_transducer(\n        encoder=str(model/\'encoder.onnx\'),decoder=str(model/\'decoder.onnx\'),joiner=str(model/\'joiner.onnx\'),\n        tokens=str(model/\'tokens.txt\'),num_threads=threads,sample_rate=16000,feature_dim=80,\n        decoding_method=\'greedy_search\',provider=\'cpu\')\n    started=time.monotonic()\n    with cache_path.open(\'a\',encoding=\'utf-8\') as output:\n        for index,name in enumerate(audio,1):\n            path=(root/split/name).resolve();assert path.is_relative_to((root/split).resolve())\n            audio_sha=digest(path)\n            if name in cache:\n                assert cache[name][\'audio_sha256\']==audio_sha,\'Audio changed after transcription\'\n                continue\n            tick=time.monotonic()\n            try:\n                samples,sample_rate=sf.read(path,dtype=\'float32\',always_2d=True)\n                samples=samples.mean(axis=1)\n                assert len(samples)>0 and np.isfinite(samples).all(),\'Empty or non-finite waveform\'\n                duration=len(samples)/sample_rate\n                if sample_rate!=16000:\n                    factor=math.gcd(sample_rate,16000)\n                    samples=resample_poly(samples,16000//factor,sample_rate//factor)\n                stream=recognizer.create_stream()\n                stream.accept_waveform(16000,np.ascontiguousarray(samples,dtype=np.float32))\n                recognizer.decode_stream(stream)\n                record={\'audio\':name,\'text\':stream.result.text.strip(),\'audio_sha256\':audio_sha,\n                    \'model_manifest_sha256\':fingerprint,\'elapsed_s\':time.monotonic()-tick,\'duration_s\':duration,\n                    \'source\':\'supplied_asr_model\',\'provider\':\'cpu\'}\n            except Exception as exc:\n                (work/\'asr_error.json\').write_text(json.dumps({\'audio\':name,\'error\':str(exc)}))\n                raise\n            output.write(json.dumps(record,ensure_ascii=False)+\'\\n\');output.flush();cache[name]=record\n            print(json.dumps({\'stage\':\'asr\',\'completed\':index,\'total_audio\':len(audio),\n                \'validation_samples\':len(selected),\'audio\':name,\'elapsed_s\':record[\'elapsed_s\']},ensure_ascii=False),flush=True)\n    assert set(audio)<=cache.keys()\n    transcripts={name:cache[name][\'text\'] for name in audio}\n    (work/\'asr_transcripts.json\').write_text(json.dumps(transcripts,ensure_ascii=False,indent=2))\n    report={\'status\':\'complete\',\'validation_samples\':len(selected),\'unique_audio\':len(audio),\n        \'current_and_history_audio\':True,\'oracle_fallback\':False,\'training\':False,\n        \'empty_transcripts\':sum(not value.strip() for value in transcripts.values()),\n        \'wall_s_this_run\':time.monotonic()-started,\n        \'audio_duration_s\':sum(cache[name][\'duration_s\'] for name in audio),\n        \'initializer_elements\':sum(info.get(\'initializer_elements\',0) for info in manifest[\'files\'].values()),\n        \'model_manifest_sha256\':fingerprint,\'provider\':\'cpu\',\'model_type\':\'zipformer2\'}\n    (work/\'asr_report.json\').write_text(json.dumps(report,indent=2))\n    print(\'ASR_COMPLETE\',json.dumps(report),flush=True)\n    return report\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser()\n    for name in (\'data-root\',\'work\',\'model-dir\'):p.add_argument(\'--\'+name,required=True)\n    p.add_argument(\'--split\',choices=[\'train\',\'public_test\'],default=\'train\')\n    p.add_argument(\'--limit\',type=int);p.add_argument(\'--threads\',type=int,default=4)\n    a=p.parse_args();transcribe(a.data_root,a.work,a.model_dir,a.limit,a.threads,a.split)\n'}
SOURCE_SHA='352d50d770d0f61d1e67bbd9da33c2e1040eb8e39a2dff227d9ed9c10177b299'
assert hashlib.sha256(json.dumps(PAYLOAD,sort_keys=True).encode()).hexdigest()==SOURCE_SHA
for name,content in PAYLOAD.items():
    path=SOURCE/name;path.parent.mkdir(parents=True,exist_ok=True);path.write_text(content)
assert hashlib.sha256((SOURCE/'split.txt').read_bytes()).hexdigest()=='631f8cfab61ddde833477f743cf1a65a9012526e0178e97046a36ee864e3d505'


In [ ]:
sys.path.insert(0,str(SOURCE))
from umilife.paths import find_data_root
DATA_ROOT,archive=find_data_root('/kaggle/input')
if DATA_ROOT is None:
    destination=Path('/tmp/v6_competition_data');destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        assert all((destination/m.filename).resolve().is_relative_to(destination.resolve()) for m in z.infolist())
        z.extractall(destination)
    DATA_ROOT,_=find_data_root(destination)
assert hashlib.sha256((DATA_ROOT/'train/train.jsonl').read_bytes()).hexdigest()=='24c42c79fbe8db932fe9817d041c0d0ca422ac8177b413cc1be0cda1fd8941a8'
def input_dir(slug,marker):
    candidates=[Path('/kaggle/input')/slug,Path('/kaggle/input/datasets/cymeu4l0t')/slug]
    return next((p for p in candidates if (p/marker).is_file()),None)
ADAPTERS=input_dir('lifemate-v6-completed-adapters','checkpoint_manifest.json')
ASR_MODEL=input_dir('lifemate-supplied-zipformer-asr','model_manifest.json')
assert ADAPTERS is not None and ASR_MODEL is not None,'Attach completed v6 adapters and supplied ASR datasets'
manifest=json.loads((ADAPTERS/'checkpoint_manifest.json').read_text())
assert manifest['version']==6 and manifest['complete_roles']==['global','planner']
missing=[name for name in manifest['files'] if not (ADAPTERS/name).is_file()]
assert not missing, f'Incomplete checkpoint input: {missing}. Update lifemate-v6-completed-adapters to dataset version 2 or later.'
print('Checkpoint input:',ADAPTERS,'Files:',len(manifest['files']))
for name,expected in manifest['files'].items():assert hashlib.sha256((ADAPTERS/name).read_bytes()).hexdigest()==expected,name
from umilife.prompt_policy import PROMPTS,digest,POLICY_VERSION
for role in ('global','planner'):
    meta=json.loads((ADAPTERS/role/'lineage.json').read_text())
    assert meta['prompt_sha256']==digest(PROMPTS) and meta['model']==MODEL and meta['revision']==REVISION
config={'mode':MODE,'model':MODEL,'revision':REVISION,'source_sha':SOURCE_SHA,
        'checkpoint_manifest_sha':hashlib.sha256((ADAPTERS/'checkpoint_manifest.json').read_bytes()).hexdigest(),
        'prompt_sha':digest(PROMPTS),'policy':POLICY_VERSION,'batch_size':4,'asr':True}
if (WORK/'config.json').exists():assert json.loads((WORK/'config.json').read_text())==config,'Different run settings; use a fresh RUN_NAME'
else:(WORK/'config.json').write_text(json.dumps(config,indent=2))
print('Checkpoint roles:',manifest['complete_roles'],'Mode:',MODE)


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','--quiet','uv==0.9.10'],check=True)
UV=[sys.executable,'-m','uv'];os.environ['UV_NATIVE_TLS']='true'
def runtime(name,lock,gpus):
    path=Path('/tmp')/name;python=str(path/'bin/python')
    if not Path(python).exists():subprocess.run([*UV,'venv','--python','3.12','--python-preference','only-managed','--seed',str(path)],check=True)
    subprocess.run([*UV,'pip','sync','--index-strategy','unsafe-best-match','--python',python,str(SOURCE/lock)],check=True)
    subprocess.run([*UV,'pip','check','--python',python],check=True)
    purelib=Path(subprocess.check_output([python,'-I','-c',"import sysconfig;print(sysconfig.get_path('purelib'))"],text=True).strip())
    env=os.environ.copy()
    for key in ('PYTHONPATH','PYTHONHOME'):env.pop(key,None)
    env.update(CUDA_VISIBLE_DEVICES=gpus,PYTHONNOUSERSITE='1',TOKENIZERS_PARALLELISM='false',
        VLLM_USE_V1='0',VLLM_ATTENTION_BACKEND='XFORMERS',VLLM_WORKER_MULTIPROC_METHOD='spawn',
        VLLM_USAGE_STATS_ENABLED='0',DO_NOT_TRACK='1',HF_HOME='/tmp/lifemate_hf_cache')
    env['LD_LIBRARY_PATH']=':'.join([str(p) for p in sorted((purelib/'nvidia').glob('*/lib'))]+[os.environ.get('LD_LIBRARY_PATH','')])
    return python,env
INFER_PYTHON,INFER_ENV=runtime('lifemate_vllm092_env','requirements_infer_2gpu.lock','0')
ASR_PYTHON,ASR_ENV=runtime('lifemate_supplied_asr_env','requirements_asr_2gpu.lock','')
subprocess.run([INFER_PYTHON,'-c',"import torch,vllm;assert torch.cuda.is_available();assert vllm.__version__=='0.9.2'"],env=INFER_ENV,check=True)


In [ ]:
def run_split(name):
    output=WORK/name;output.mkdir(exist_ok=True)
    if name=='validation':ids=(SOURCE/'split.txt').read_text().splitlines();assert len(ids)==100;split='train'
    else:
        ids=[json.loads(line)['id'] for line in (DATA_ROOT/'public_test/public_test.jsonl').open()];split='public_test'
    assert len(ids)==len(set(ids))
    (output/'split.txt').write_text('\n'.join(ids)+'\n')
    subprocess.run([ASR_PYTHON,'-u','-m','umilife.transcribe_supplied','--data-root',str(DATA_ROOT),
        '--work',str(output),'--model-dir',str(ASR_MODEL),'--split',split],cwd=SOURCE,env=ASR_ENV,check=True)
    subprocess.run([INFER_PYTHON,'-u','-m','umilife.infer_2gpu','--data-root',str(DATA_ROOT),
        '--work',str(output),'--model',MODEL,'--revision',REVISION,'--adapters',str(ADAPTERS),
        '--ids-file',str(output/'split.txt'),'--split',name,'--mode',MODE,
        '--deadline-monotonic',str(SESSION_STARTED+SESSION_HOURS*3600)],cwd=SOURCE,env=INFER_ENV,check=True)
    report=json.loads((output/'report.json').read_text());assert report['samples']==len(ids)
    return report
validation=run_split('validation')
assert validation['samples']==100
print('VALIDATION:',json.dumps(validation['metrics'],ensure_ascii=False,indent=2))


In [ ]:
# This cell executes only after validation completes. No public-test gold data is used.
assert (WORK/'validation/report.json').exists()
public_test=run_split('public_test')
print('Public-test samples:',public_test['samples'])
print('Submission:',WORK/'public_test/submission.jsonl')
